# Transient Percolation Zone Analysis

This notebook explores how the **percolation zone** has migrated over the course of the IMAU-FDM simulation by analysing annual surface melt and runoff events at each grid point.

Three analyses are available for **any registered variable** (currently `surfmelt` and `Runoff`):

1. **Event frequency map** — how many years did each pixel exceed the annual threshold? Covers either the full simulation or a user-defined trailing window.
2. **Event frequency histogram** — side-by-side comparison of the per-pixel event count distribution between two time periods.
3. **Zone migration map** — classify each pixel as low-event or high-event in an early and a late window, then map which pixels have transitioned.

All defaults (paths, thresholds, windows) live in `config.py` and can be overridden inline in any cell.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, '.')
import config
from transient_percolation_analysis import (
    map_melt_frequency,
    plot_melt_histogram,
    plot_percolation_migration,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12

## Configuration

Edit `config.py` to change paths and defaults.  
All function arguments below can also be overridden inline without touching the config file.

In [ ]:
# Confirm the variable files exist before running the analyses
for var, path in config.VARIABLE_FILES.items():
    status = '✓' if path.exists() else '✗  NOT FOUND'
    print(f'  [{status}]  {var:10s}  {path.name}')

print(f'\nEvent thresholds : {config.EVENT_THRESHOLDS}')
print(f'Period 1 / 2     : {config.PERIOD_1}  /  {config.PERIOD_2}')
print(f'Early / late win : {config.EARLY_WINDOW}  /  {config.LATE_WINDOW}')
print(f'Dry / perc limit : ≤{config.DRY_EVENTS_MAX}  /  ≥{config.PERCOLATION_EVENTS_MIN} event years')

---
## Surface melt analyses

### Analysis 1 – Event frequency map

Each pixel is coloured by how many years its annual surface melt exceeded
`config.EVENT_THRESHOLDS['surfmelt']` (default 10 mm w.e.).

In [ ]:
# 1a. All years in the simulation
count_all, ax = map_melt_frequency()
plt.tight_layout()
plt.show()

In [ ]:
# 1b. Last N years only – change years_back to taste
YEARS_BACK = 10

count_recent, ax = map_melt_frequency(years_back=YEARS_BACK)
plt.tight_layout()
plt.show()

In [ ]:
# 1c. Side-by-side: full simulation vs. last N years
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

map_melt_frequency(ax=axes[0])
map_melt_frequency(years_back=YEARS_BACK, ax=axes[1])

plt.tight_layout()
plt.show()

### Analysis 2 – Period comparison histogram

Grouped bar chart showing how the distribution of per-pixel surface melt event counts has
changed between two periods.  A rightward shift in the later period indicates more
widespread or more frequent melt.

In [ ]:
# Uses config.PERIOD_1 and config.PERIOD_2 by default
ax = plot_melt_histogram()
plt.tight_layout()
plt.show()

In [ ]:
# Custom periods – override inline
ax = plot_melt_histogram(
    period1=(1950, 1980),
    period2=(1993, 2023),
)
plt.tight_layout()
plt.show()

### Analysis 3 – Percolation zone migration map

Compares each pixel's surface melt event count in an **early window** against a **late window**.
Pixels with ≤ `DRY_EVENTS_MAX` event years are classed as dry facies; pixels with
≥ `PERCOLATION_EVENTS_MIN` event years are classed as percolation zone.

| Colour | Category |
|--------|----------|
| Blue   | Remained dry facies in both windows |
| Orange | Remained in the percolation zone in both windows |
| **Red** | **Expanded: dry → percolation** ← main signal |
| Purple | Contracted: percolation → dry |
| White  | Off-ice, or ambiguous (event count falls between the two thresholds) |

In [ ]:
# Uses config defaults (EARLY_WINDOW, LATE_WINDOW, DRY_EVENTS_MAX, PERCOLATION_EVENTS_MIN)
classification, ax = plot_percolation_migration()
plt.tight_layout()
plt.show()

In [ ]:
# Custom windows and thresholds
classification, ax = plot_percolation_migration(
    early_window=(1950, 1969),
    late_window=(2003, 2022),
    dry_events_max=2,
    percolation_events_min=3,
)
plt.tight_layout()
plt.show()

In [ ]:
# How many pixels transitioned in each category?
import numpy as np

labels = {
    1: 'Remained dry',
    2: 'Remained percolation',
    3: 'Expanded (dry → percolation)',
    4: 'Contracted (percolation → dry)',
}
for cat, label in labels.items():
    n = int((classification == cat).sum())
    print(f'  {label:35s}: {n:6d} pixels')

---
## Combined figure

All three analyses in one overview figure.

In [ ]:
fig = plt.figure(figsize=(20, 10))

# Left: melt frequency (all time)
ax1 = fig.add_subplot(1, 3, 1)
map_melt_frequency(ax=ax1)

# Middle: percolation migration
ax2 = fig.add_subplot(1, 3, 2)
plot_percolation_migration(ax=ax2)

# Right: histogram
ax3 = fig.add_subplot(1, 3, 3)
plot_melt_histogram(ax=ax3)

plt.tight_layout()
plt.show()

---
## Runoff analyses

The same three analyses repeated for **runoff** (`variable='Runoff'`).

The event threshold for runoff is set separately in `config.EVENT_THRESHOLDS['Runoff']`
(default 1.0 mm w.e. yr⁻¹) and can be overridden inline with `threshold=`.

### Analysis 1 – Event frequency map

Each pixel is coloured by how many years its annual runoff exceeded
`config.EVENT_THRESHOLDS['Runoff']` (default 1 mm w.e.).

In [ ]:
# Side-by-side: full simulation and last 10 years
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

map_melt_frequency(variable='Runoff', ax=axes[0])
map_melt_frequency(variable='Runoff', years_back=10, ax=axes[1])

plt.tight_layout()
plt.show()

### Analysis 2 – Period comparison histogram

Grouped bar chart comparing runoff event counts between the two configured periods.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_melt_histogram(variable='Runoff', ax=ax)
plt.tight_layout()
plt.show()

### Analysis 3 – Zone migration map

Pixels with ≤ `DRY_EVENTS_MAX` runoff event years are classed as low-runoff;
pixels with ≥ `PERCOLATION_EVENTS_MIN` are classed as high-runoff.

| Colour | Category |
|--------|----------|
| Blue   | Remained low runoff in both windows |
| Orange | Remained high runoff in both windows |
| **Red** | **Expanded: low → high runoff** |
| Purple | Contracted: high → low runoff |

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
classification_runoff, ax = plot_percolation_migration(variable='Runoff', ax=ax)
plt.tight_layout()
plt.show()

# Pixel counts per category
labels = {
    1: 'Remained low runoff',
    2: 'Remained high runoff',
    3: 'Expanded (low → high runoff)',
    4: 'Contracted (high → low runoff)',
}
for cat, label in labels.items():
    n = int((classification_runoff == cat).sum())
    print(f'  {label:35s}: {n:6d} pixels')

### Side-by-side comparison: surfmelt vs runoff migration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

plot_percolation_migration(variable='surfmelt', ax=axes[0])
plot_percolation_migration(variable='Runoff',   ax=axes[1])

plt.tight_layout()
plt.show()